# 워크북 3 — 강의자 완성본
위에서 아래로 실습 0~6을 실행합니다. 참조 PDF에서 하나를 내려받아 업로드하세요.
이 버전은 API 없는 무료 원문 검색 채팅 UI이며 LLM 생성형 답변은 포함하지 않습니다.


## 실습 0. 라이브러리 설치
셀을 실행하고 출력·UI 동작을 함께 확인하세요.


In [ ]:
%pip -q install pypdf scikit-learn gradio pandas


## 실습 1. PDF 한 개 업로드
셀을 실행하고 출력·UI 동작을 함께 확인하세요.


In [ ]:
from google.colab import files
from pypdf import PdfReader
uploaded = files.upload()
pdf_names = [n for n in uploaded if n.lower().endswith(".pdf")]
assert len(pdf_names) == 1, "PDF 한 개만 선택하세요."
pdf_name = pdf_names[0]
reader = PdfReader(pdf_name)
records = []
for page_no, page in enumerate(reader.pages, 1):
    text = (page.extract_text() or "").strip()
    if text:
        records.append({"page": page_no, "text": text})
assert records, "추출할 문자가 없습니다. 이미지 PDF는 OCR이 필요합니다."
print("선택 문서:", pdf_name, "/ 텍스트 페이지:", len(records))


## 실습 2. 청킹과 검색 인덱스
셀을 실행하고 출력·UI 동작을 함께 확인하세요.


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def chunk_words(text, size=90, overlap=20):
    if size <= overlap or overlap < 0:
        raise ValueError("size > overlap >= 0 이어야 합니다.")
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i+size]) for i in range(0, len(words), step)]

chunks = [
    {"page": item["page"], "text": part}
    for item in records
    for part in chunk_words(item["text"], 90, 20)
]
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
index = vectorizer.fit_transform([item["text"] for item in chunks])
print("청크:", len(chunks), "/ 벡터 행렬:", index.shape)


## 실습 3. Top-k 유사도 검색
셀을 실행하고 출력·UI 동작을 함께 확인하세요.


In [ ]:
def search_pdf(question, top_k=3, threshold=0.05):
    if not question.strip():
        return []
    q = vectorizer.transform([question])
    scores = cosine_similarity(q, index).ravel()
    order = np.argsort(scores)[::-1][:int(top_k)]
    return [
        {**chunks[int(i)], "score": float(scores[int(i)])}
        for i in order if scores[int(i)] >= threshold
    ]

hits = search_pdf("이 문서의 목적은 무엇인가요?")
for hit in hits:
    print(f"PDF p.{hit['page']} | {hit['score']:.3f}")


## 실습 4. 출처 표시 답변
셀을 실행하고 출력·UI 동작을 함께 확인하세요.


In [ ]:
def build_answer(question, top_k, threshold):
    hits = search_pdf(question, top_k, threshold)
    if not hits:
        return "PDF에서 관련 근거를 찾지 못했습니다.", "검색 결과 없음"
    best = hits[0]
    answer = (
        f"📌 PDF p.{best['page']} 검색 원문\n\n"
        f"{best['text']}\n\n"
        "※ LLM이 작성한 답변이 아니라 검색된 원문입니다."
    )
    source_text = "\n\n".join(
        f"[{i}] p.{h['page']} / 점수 {h['score']:.3f}\n{h['text']}"
        for i, h in enumerate(hits, 1)
    )
    return answer, source_text

print(build_answer("이 문서의 목적은 무엇인가요?", 3, 0.05)[0][:160])


## 실습 5. Gradio 채팅 UI·슬라이더·초기화
셀을 실행하고 출력·UI 동작을 함께 확인하세요.


In [ ]:
import gradio as gr

def on_send(message, history, top_k, threshold):
    history = list(history or [])
    if not message.strip():
        return history, "", "질문을 입력하세요."
    answer, sources = build_answer(message, top_k, threshold)
    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})
    return history, "", sources

with gr.Blocks(title="PDF 검색 도우미") as demo:
    gr.Markdown("# 📚 PDF 근거 검색 챗봇\n선택 문서의 실제 내용을 검색합니다.")
    gr.Markdown(f"**선택 문서:** {pdf_name}")
    with gr.Row():
        with gr.Column(scale=3):
            chat = gr.Chatbot(label="대화 기록", type="messages", height=400)
            question = gr.Textbox(label="질문")
            send = gr.Button("검색하기", variant="primary")
            clear = gr.Button("대화 초기화")
        with gr.Column(scale=2):
            top_k = gr.Slider(1, 5, value=3, step=1, label="검색 결과 수")
            threshold = gr.Slider(0, 1, value=0.05, step=0.01, label="최소 유사도")
            sources = gr.Textbox(label="문서 근거와 페이지", lines=16)
    send.click(on_send, [question, chat, top_k, threshold], [chat, question, sources])
    question.submit(on_send, [question, chat, top_k, threshold], [chat, question, sources])
    clear.click(lambda: ([], "", ""), outputs=[chat, question, sources])

demo.launch(share=False)


## 실습 6. 결과표 작성
셀을 실행하고 출력·UI 동작을 함께 확인하세요.


In [ ]:
import pandas as pd
questions = [
    "이 문서의 목적은 무엇인가요?",
    "주요 절차는 무엇인가요?",
    "화성 탐사선 발사일은 언제인가요?"
]
rows = []
for q in questions:
    hits = search_pdf(q, 3, 0.05)
    rows.append({
        "질문": q,
        "검색 개수": len(hits),
        "근거 페이지": hits[0]["page"] if hits else "없음",
        "최고 점수": round(hits[0]["score"], 3) if hits else 0
    })
display(pd.DataFrame(rows))
